# EarthDaily Agriculture — KPI Functions Dev

Validation notebook for `filter_timeseries_kpi` and the KPI extractor wiring.

1. Extract a 10‑day MRTS **smoothed** window for one entity, current season + last 5 years
2. Display the per‑year curves overlaid so you can eyeball values
3. Run **every** supported KPI aggregation against the same series and collect the results in one table

Use this notebook to spot regressions whenever `filter_timeseries_kpi` or `validate_kpi_filter` change.

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

## Step 1 — Init

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
from earthdaily.agriculture.extractors.VTS_functions import MRTSExtractor

manager = WorkflowManager('prod')

## Step 2 — Load entities and pick a test entity

In [ ]:
manager.load_seasonfields(
    sowing_date_gte='2025-07-01',
    crop_id='WINTER_OSR',
)

row = manager.sfd_list.iloc[4].to_dict()
print(f"Test entity: {row['id']}")

## Step 3 — Extract a 10‑day MRTS smoothed window with 5 years of history

10 days is short on purpose — small enough to hand‑check every value, long enough to give the rolling and percentile aggregations something to chew on.

In [ ]:
import pandas as pd

start_date = '2026-02-19'
end_date   = '2026-02-28'

mrts = MRTSExtractor(
    bearer_token=manager.bearer_token,
    token_expiration=manager.token_expiration,
    config=manager.config,
    workflow_ref=manager,
)

column_mapping = {'crop': 'crop.id', 'start_date': 'sowingDate'}

mrts.setup_mrts_parameters(
    start_date=start_date,
    end_date=end_date,
    vegetation_index='NDVI',
    mode='full',
    historical_years=5,
    column_mapping=column_mapping,
    kpi_filter=None,
)

raw_result = mrts.process_single_entity_mrts(row=row)
ts_df = raw_result['data']
if raw_result.get('error'):
    print(f"Extraction error: {raw_result['error']}")
else:
    ts_df['date'] = pd.to_datetime(ts_df['date'])
    ts_df = ts_df.sort_values('date').reset_index(drop=True)
    print(f"Shape:        {ts_df.shape}")
    print(f"Date range:   {ts_df['date'].min().date()} → {ts_df['date'].max().date()}")
    print(f"Years:        {sorted(ts_df['date'].dt.year.unique().tolist())}")
    print(f"Columns:      {ts_df.columns.tolist()}")
ts_df.head()

## Step 4 — Display the per‑year curves

Line chart with one curve per year, x‑axis = `month-day` so the 10‑day windows from each year stack on top of each other. Current year is drawn thicker in red. The pivot table below makes it easy to compare values cell‑by‑cell.

In [ ]:
import plotly.graph_objects as go

plot_df = ts_df[['date', 'smoothed_value']].dropna().copy()
plot_df['year'] = plot_df['date'].dt.year
plot_df['md']   = plot_df['date'].dt.strftime('%m-%d')
current_year = plot_df['date'].max().year

fig = go.Figure()
for y, sub in plot_df.groupby('year'):
    sub = sub.sort_values('md')
    is_current = (y == current_year)
    fig.add_scatter(
        x=sub['md'], y=sub['smoothed_value'],
        mode='lines+markers',
        name=f"{y}{' (current)' if is_current else ''}",
        line=dict(width=3 if is_current else 1, color='#d62728' if is_current else None),
    )
fig.update_layout(
    title='NDVI smoothed — current season vs. last 5 years',
    xaxis_title='month-day', yaxis_title='NDVI',
    height=420, template='plotly_white',
    legend=dict(orientation='h', y=1.06, yanchor='bottom'),
)
fig.show()

pivot = (
    plot_df
    .pivot_table(index='md', columns='year', values='smoothed_value')
    .sort_index()
    .round(4)
)
print('Per-year values (rows = month-day, columns = year):')
display(pivot)

## Step 5 — Run every supported KPI aggregation

Driven by `KPI_AGGREGATION_RULES` so any aggregation added later is automatically exercised. Standalone `filter_timeseries_kpi` calls — no extra API round‑trips.

The `DEFAULTS` table picks reasonable values for `threshold` / `window` so each aggregation returns something non‑trivial out of the box. Tweak any row to probe edge cases.

In [ ]:
import pandas as pd
from earthdaily.agriculture.core.api_utils import KPI_AGGREGATION_RULES, filter_timeseries_kpi

v_min = float(ts_df['smoothed_value'].min())
v_max = float(ts_df['smoothed_value'].max())
mid   = (v_min + v_max) / 2
q25   = v_min + 0.25 * (v_max - v_min)
q75   = v_min + 0.75 * (v_max - v_min)

# Sensible defaults so every KPI returns something interesting on a 10-day window
DEFAULTS = {
    'accumulation':     {},
    'top_accumulation': {'threshold': 3},
    'average':          {},
    'max':              {},
    'min':              {},
    'std':              {},
    'count_gt':         {'threshold': mid},
    'count_lt':         {'threshold': mid},
    'count_between':    {'threshold': (q25, q75)},
    'rolling_avg':      {'window': 3},
    'rolling_avg_gt':   {'window': 3, 'threshold': mid},
    'rolling_avg_lt':   {'window': 3, 'threshold': mid},
    'percentile':       {'threshold': 90},
    'percentile_gt':    {'threshold': 90},
    'percentile_lt':    {'threshold': 10},
}
assert set(DEFAULTS.keys()) == set(KPI_AGGREGATION_RULES.keys()), \
    'DEFAULTS out of sync with KPI_AGGREGATION_RULES — update this cell.'

rows = []
for agg, kw in DEFAULTS.items():
    try:
        result = filter_timeseries_kpi(
            timeseries_df=ts_df,
            start_date=start_date,
            end_date=end_date,
            kpi_name=agg,
            aggregation=agg,
            years='ALL',
            date_column='date',
            value_column='smoothed_value',
            **kw,
        )
        rows.append({
            'aggregation':      agg,
            'params':           kw or '',
            'current':          result['current_period']['value'],
            'records':          result['current_period']['num_records'],
            'historical_avg':   result['historical_avg']['value'],
            'historical_years': result['historical_avg']['num_years'],
            'difference':       result['comparison']['difference'],
            'percent_change':   result['comparison']['percent_change'],
            'error':            None,
        })
    except Exception as exc:
        rows.append({'aggregation': agg, 'params': kw or '', 'error': str(exc)})

results_df = pd.DataFrame(rows)
results_df

## Validation checklist

- For each aggregation the `current` value should match what you can compute by hand from the 10‑day curve in Step 4.
- `historical_avg` is the mean of the same KPI computed across the previous years' matching 10‑day windows (Step 4 pivot table makes this easy to verify).
- `rolling_avg*` aggregations need a window strictly smaller than the number of records — with only 10 points, `window=3` is a reasonable demo; bumping the window changes the count sharply, which is the whole point.
- `percentile_gt` / `percentile_lt` thresholds are *percentile ranks* (0–100), not value cutoffs. With only 10 points a `threshold=90` keeps roughly the top one or two values — expect small but non‑zero counts.
- If any row in the table shows an `error`, that's a regression: trace it to either the validator (`api_utils.validate_kpi_filter`) or the compute branch in `filter_timeseries_kpi.compute_kpi`.